In [20]:
# Install the same dependencies used by the backend codebase.
%pip install -r ../requirements.txt


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [28]:
from pystac_client import Client
import json

# 1. Connect to the Copernicus Data Space STAC Catalog
STAC_URL = "https://catalogue.dataspace.copernicus.eu/stac"
catalog = Client.open(STAC_URL)

# 2. Define your search parameters
# Bounding box for Ilfov County / Bucharest [min_lon, min_lat, max_lon, max_lat]
bbox = [25.85, 44.33, 26.27, 44.65]

# Time window (Adjust to your preferred historical window)
time_range = "2026-04-01/2026-04-24"

print("Searching Copernicus Data Space for Sentinel-2 passes over Ilfov...")

# 3. Execute the search
search = catalog.search(
    collections=["sentinel-2-l2a"],     # L2A is atmospherically corrected (ready for ML)
    bbox=bbox,
    datetime=time_range,
    query={"eo:cloud_cover": {"lt": 50}} # Filter: Less than 20% cloud cover
)

# 4. Extract the results
items = list(search.items())
print(f"Found {len(items)} cloud-free satellite passes!\n")

# 5. Inspect the most recent pass
if len(items) > 0:
    latest_item = items[0]

    print(f"--- LATEST PASS DETAILS ---")
    print(f"Date & Time: {latest_item.datetime}")
    print(f"Cloud Cover: {latest_item.properties.get('eo:cloud_cover')}%")
    print(f"Platform: {latest_item.properties.get('platform')}")

    print(f"\n--- DATA ASSET LINKS ---")
    print("These URLs point directly to the cloud-optimized GeoTIFFs (COGs).")

    # Sentinel-2 has many bands. Let's print the links for a few crucial ones.
    # B04 = Red, B08 = Near Infrared (Used together to calculate Vegetation/NDVI)

    assets_of_interest = ['B04', 'B08', 'SCL'] # SCL is the Scene Classification Layer (masks out clouds/water)

    for asset_key in assets_of_interest:
        if asset_key in latest_item.assets:
            print(f"Band {asset_key}: {latest_item.assets[asset_key].href}")

else:
    print("No data found matching those parameters.")

Searching Copernicus Data Space for Sentinel-2 passes over Ilfov...
Found 8 cloud-free satellite passes!

--- LATEST PASS DETAILS ---
Date & Time: 2026-04-24 09:20:51.024000+00:00
Cloud Cover: 38.07%
Platform: sentinel-2a

--- DATA ASSET LINKS ---
These URLs point directly to the cloud-optimized GeoTIFFs (COGs).


### 1. Authenticate with Copernicus Data Space Ecosystem (CDSE)
To download high-resolution satellite data or stream it directly into our ML pipeline, we need an OAuth2 access token.

In [33]:
import os
import requests
from getpass import getpass
from dotenv import load_dotenv

# Load notebook-local secrets from backend/notebooks/.env
load_dotenv('.env', override=True)

def _clean_env(name: str) -> str | None:
    value = os.getenv(name)
    if value is None:
        return None
    value = value.strip()
    return value if value else None

def get_access_token(interactive_fallback: bool = True):
    """Get CDSE token.

    Priority:
    1) Username/password flow (usually needed for Product zip download)
    2) Client credentials flow (works for many API calls/search)
    """
    username = _clean_env('COPERNICUS_USERNAME')
    password = _clean_env('COPERNICUS_PASSWORD')
    client_id = _clean_env('COPERNICUS_CLIENT_ID')
    client_secret = _clean_env('COPERNICUS_CLIENT_SECRET')

    # Ask interactively if user creds are missing.
    if interactive_fallback and (not username or not password):
        print('Username/password not found in .env. Enter CDSE account credentials for Product download.')
        if not username:
            typed_user = input('CDSE username/email: ').strip()
            username = typed_user or None
        if not password:
            typed_pass = getpass('CDSE password: ').strip()
            password = typed_pass or None

    auth_server_url = 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token'

    # Prefer user credentials when present.
    if username and password:
        data = {
            'client_id': 'cdse-public',
            'username': username,
            'password': password,
            'grant_type': 'password',
        }
    elif client_id and client_secret:
        data = {
            'client_id': client_id,
            'client_secret': client_secret,
            'grant_type': 'client_credentials',
        }
    else:
        print('Error: Set COPERNICUS_USERNAME/COPERNICUS_PASSWORD or COPERNICUS_CLIENT_ID/COPERNICUS_CLIENT_SECRET in notebooks/.env')
        return None

    response = requests.post(auth_server_url, data=data, timeout=60)
    response.raise_for_status()
    return response.json().get('access_token')

access_token = get_access_token()
if access_token:
    print('Successfully authenticated with CDSE.')

Successfully authenticated with CDSE.


### 2. Search for Sentinel-1 (SAR) Radar Data
Radar is crucial for FloodWise because it 'sees' through clouds. We look for 'GRD' (Ground Range Detected) products which are best for water extent mapping.

In [30]:
s1_search = catalog.search(
    collections=["sentinel-1-grd"],
    bbox=bbox,
    datetime=time_range,
    query={"sar:instrument_mode": {"eq": "IW"}, "sar:product_type": {"eq": "GRD"}}
)

s1_items = list(s1_search.items())
print(f"Found {len(s1_items)} Sentinel-1 radar passes for the period.")

if s1_items:
    print(f"Latest Radar ID: {s1_items[0].id}")

Found 0 Sentinel-1 radar passes for the period.


### 3. Feature Extraction: Calculate NDVI
NDVI measures vegetation health. Low NDVI in areas that should be green can indicate saturated soil or standing water. We use `rasterio` to stream the specific bands (Red and NIR) directly from the cloud.

In [31]:
# Dependencies are already installed from ../requirements.txt in Cell 1.
# This cell can be kept as a quick import check.
import rasterio
print('rasterio import ok')

rasterio import ok


In [25]:
if items:
    print(f"Available assets for this item: {list(items[0].assets.keys())}")
    # Displaying titles to help map them to bands
    for key, asset in items[0].assets.items():
        print(f"{key}: {asset.title}")

Available assets for this item: ['AOT_10m', 'AOT_20m', 'AOT_60m', 'B01_20m', 'B01_60m', 'B02_10m', 'B02_20m', 'B02_60m', 'B03_10m', 'B03_20m', 'B03_60m', 'B04_10m', 'B04_20m', 'B04_60m', 'B05_20m', 'B05_60m', 'B06_20m', 'B06_60m', 'B07_20m', 'B07_60m', 'B08_10m', 'B09_60m', 'B11_20m', 'B11_60m', 'B12_20m', 'B12_60m', 'B8A_20m', 'B8A_60m', 'CLD_20m', 'CLD_60m', 'Product', 'SCL_20m', 'SCL_60m', 'SNW_20m', 'SNW_60m', 'TCI_10m', 'TCI_20m', 'TCI_60m', 'WVP_10m', 'WVP_20m', 'WVP_60m', 'thumbnail', 'safe_manifest', 'granule_metadata', 'inspire_metadata', 'product_metadata', 'datastrip_metadata']
AOT_10m: Aerosol optical thickness (AOT) - 10m
AOT_20m: Aerosol optical thickness (AOT) - 20m
AOT_60m: Aerosol optical thickness (AOT) - 60m
B01_20m: Coastal aerosol (band 1) - 20m
B01_60m: Coastal aerosol (band 1) - 60m
B02_10m: Blue (band 2) - 10m
B02_20m: Blue (band 2) - 20m
B02_60m: Blue (band 2) - 60m
B03_10m: Green (band 3) - 10m
B03_20m: Green (band 3) - 20m
B03_60m: Green (band 3) - 60m
B04_10

## 4. Download, Extract, and Run NDWI
Run the next three cells in order after authentication succeeds and `latest_item` is available.

In [37]:
from pathlib import Path
import json
import re
import base64
import requests

# Download the zipped product referenced by the selected STAC item.
download_dir = Path("../data/raw")
download_dir.mkdir(parents=True, exist_ok=True)

product_href = latest_item.assets["Product"].href
zip_path = download_dir / f"{latest_item.id}.zip"

def _extract_product_id(href: str) -> str | None:
    match = re.search(r"Products\(([^)]+)\)", href)
    if not match:
        return None
    return match.group(1).strip("\"'")

def _jwt_payload(token: str) -> dict:
    parts = token.split('.') if isinstance(token, str) else []
    if len(parts) != 3:
        return {}
    padding = '=' * (-len(parts[1]) % 4)
    try:
        return json.loads(base64.urlsafe_b64decode(parts[1] + padding).decode('utf-8'))
    except Exception:
        return {}

def download_product_zip(href: str, token: str, out_path: Path) -> str:
    """Download product zip, retrying with fresh token and zipper fallback on 401."""
    current_token = token

    # Early guard: service-account tokens often fail with 401 for product binaries.
    payload = _jwt_payload(current_token)
    preferred_username = str(payload.get('preferred_username', ''))
    if preferred_username.startswith('service-account-'):
        raise RuntimeError(
            'Current token is service-account based. Product download needs user token. '
            'Set COPERNICUS_USERNAME and COPERNICUS_PASSWORD in notebooks/.env and rerun auth cell.'
        )

    with requests.Session() as session:
        def _download(url: str, bearer: str):
            headers = {"Authorization": f"Bearer {bearer}"}
            return session.get(url, headers=headers, stream=True, timeout=600)

        response = _download(href, current_token)

        # Token can expire between auth cell and download cell.
        if response.status_code == 401:
            refreshed = get_access_token()
            if refreshed:
                current_token = refreshed
                response = _download(href, current_token)

        # Some products download more reliably from zipper endpoint.
        if response.status_code == 401:
            product_id = _extract_product_id(href)
            if product_id:
                zipper_url = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products({product_id})/$value"
                response = _download(zipper_url, current_token)

        response.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    return current_token

access_token = download_product_zip(product_href, access_token, zip_path)
print(f"Downloaded: {zip_path.resolve()}")

RuntimeError: Current token is service-account based. Product download needs user token. Set COPERNICUS_USERNAME and COPERNICUS_PASSWORD in notebooks/.env and rerun auth cell.

In [ ]:
import zipfile

# Extract zip and locate the .SAFE folder.
extract_dir = Path("../data/extracted")
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

safe_dirs = sorted(extract_dir.glob("*.SAFE"))
if not safe_dirs:
    raise RuntimeError("No .SAFE folder found after extraction.")

safe_dir = safe_dirs[-1]
print(f"SAFE folder: {safe_dir.resolve()}")

In [ ]:
import sys

# Import backend processing code from the parent folder.
sys.path.append(str(Path("..").resolve()))
from sentinel2_ndwi import run_ndwi_pipeline

output_dir = Path("../data/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

result = run_ndwi_pipeline(
    safe_dir=str(safe_dir.resolve()),
    output_path=str((output_dir / "ndwi_output.tif").resolve()),
    bbox=(25.85, 44.33, 26.27, 44.65),
    bbox_crs="EPSG:4326",
    water_threshold=0.05,
    water_mask_output_path=str((output_dir / "ndwi_water_mask.tif").resolve()),
    preview_png_output_path=str((output_dir / "ndwi_preview.png").resolve()),
)

print(result)

In [34]:
import os
import json
import base64

def _b64url_decode(payload: str) -> str:
    padding = '=' * (-len(payload) % 4)
    return base64.urlsafe_b64decode(payload + padding).decode('utf-8')

has_user = bool(os.getenv('COPERNICUS_USERNAME')) and bool(os.getenv('COPERNICUS_PASSWORD'))
has_client = bool(os.getenv('COPERNICUS_CLIENT_ID')) and bool(os.getenv('COPERNICUS_CLIENT_SECRET'))
print('Has username/password:', has_user)
print('Has client credentials:', has_client)

parts = access_token.split('.') if isinstance(access_token, str) else []
if len(parts) == 3:
    payload = json.loads(_b64url_decode(parts[1]))
    print('Token preferred_username:', payload.get('preferred_username'))
    print('Token azp (authorized party):', payload.get('azp'))
    print('Token scope:', payload.get('scope'))
else:
    print('Token format is not JWT-like')

Has username/password: False
Has client credentials: True
Token preferred_username: service-account-sh-55d49956-fc8a-45dd-87c9-0ed1bac12888
Token azp (authorized party): sh-55d49956-fc8a-45dd-87c9-0ed1bac12888
Token scope: email profile user-context
